# Chapter 3 Practice: Market Direction (messy, real-world data), preprocessing starter

**Data:**
- **S&P 500 index, daily OHLCV, 1999–2018**, downloaded from Yahoo Finance and bundled with the `arch` package (`pip install arch`).
- **VIX index, daily OHLC, 1990–today**, from the [datahub `finance-vix`](https://github.com/datasets/finance-vix) CSV on GitHub. VIX is the market's expectation of volatility over the next 30 days, which is why it's often called the "fear gauge".

**Why this is "messy" compared with `Smarket`:**
1. Two sources with **different calendars** have to be aligned.
2. There are **data-quality errors** in the raw files.
3. There are **no features and no label**. All you get is prices, and you have to build both yourself.
4. Prices are **non-stationary**, so they can't be used as features directly.
5. **Lookahead bias** is easy to introduce by accident.
6. Ordinary k-fold CV **leaks the future** into training.

**How to use this notebook:** run the preprocessing cells below and answer the 🤔 questions for yourself, writing notes in a markdown cell if that helps. Once `X_train`, `y_train_up`, etc. exist, apply the Chapter 3 steps in the empty sections at the bottom. You can work in this notebook directly or copy it to `03_messy_sp500_vix.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rc("font", size=12)
plt.rc("axes", labelsize=12, titlesize=13)

## P1. Load both sources

In [ ]:
from arch.data import sp500  # pip install arch

spx = sp500.load()
spx.head()

In [ ]:
vix_url = "https://raw.githubusercontent.com/datasets/finance-vix/main/data/vix-daily.csv"
vix_raw = pd.read_csv(vix_url, parse_dates=["DATE"], index_col="DATE")
vix_raw.tail()

## P2. Inspect: types, ranges, duplicates, calendars

In [ ]:
print("SPX:", spx.index.min().date(), "->", spx.index.max().date(), spx.shape)
print("VIX:", vix_raw.index.min().date(), "->", vix_raw.index.max().date(), vix_raw.shape)
print("duplicate dates  SPX:", spx.index.duplicated().sum(), " VIX:", vix_raw.index.duplicated().sum())
print("missing values   SPX:", spx.isna().sum().sum(), " VIX:", vix_raw.isna().sum().sum())

In [ ]:
# Only look at the overlapping period
vix = vix_raw.loc[spx.index.min():spx.index.max()]

print("Dates in SPX but not VIX:", list(spx.index.difference(vix.index).date))
print("Dates in VIX but not SPX:", list(vix.index.difference(spx.index).date))

> 🤔 **Think before moving on:** Why would the two sources disagree on which days are trading days? Look up what happened on 2004-06-11. Which calendar should be the 'master', and why?

In [ ]:
# Sanity check: in every OHLC bar, Low <= Open <= High and Low <= Close <= High must hold
bad_vix = vix[(vix["OPEN"] < vix["LOW"]) | (vix["OPEN"] > vix["HIGH"])]
print(f"VIX rows where OPEN is outside [LOW, HIGH]: {len(bad_vix)}")
bad_vix.head(10)

In [ ]:
bad_spx = spx[(spx["Open"] < spx["Low"]) | (spx["Open"] > spx["High"]) |
              (spx["Close"] < spx["Low"]) | (spx["Close"] > spx["High"])]
print(f"SPX rows breaking OHLC rules: {len(bad_spx)}")
print(f"SPX rows with zero volume:    {(spx['Volume'] == 0).sum()}")
print(f"Close != Adj Close on {(spx['Close'] != spx['Adj Close']).sum()} rows")

> 🤔 **Think before moving on:** Some VIX OPEN values are impossible. Should you fix them, drop those rows, or avoid the column? What would you lose with each option?

In [ ]:
# Calendar gaps: how many calendar days between consecutive trading days?
gaps = spx.index.to_series().diff().dt.days
print(gaps.value_counts().sort_index())
gaps[gaps >= 5]

> 🤔 **Think before moving on:** Look up what happened just before 2001-09-17, 2007-01-03 and 2012-10-31. Is the 'daily' return on those days comparable to other days?

## P3. Align the two sources on the S&P 500 calendar

In [ ]:
df = spx.join(vix[["CLOSE"]].rename(columns={"CLOSE": "VIX"}), how="left")
print("Missing VIX after join:", df["VIX"].isna().sum())
df["VIX"] = df["VIX"].ffill(limit=1)  # carry forward at most 1 day
print("Missing VIX after ffill:", df["VIX"].isna().sum())
df.head()

## P4. Prices are non-stationary, so use returns

In [ ]:
log_ret = np.log(df["Adj Close"]).diff()

fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
df["Adj Close"].plot(ax=axs[0], title="S&P 500 level (non-stationary)")
log_ret.plot(ax=axs[1], title="Daily log return (roughly stationary, but volatility clusters)", linewidth=0.5)
plt.tight_layout()
plt.show()

> 🤔 **Think before moving on:** If you used the raw price level (e.g. 2,500) as a feature, what would a tree trained on 1999–2014 do on 2015–2018, when prices are higher than anything it saw in training?

## P5. Feature engineering: use only information known at the close of day *t*

In [ ]:
px = df["Adj Close"]
feat = pd.DataFrame(index=df.index)

for k in range(5):                                 # today's return and the 4 before it
    feat[f"ret_{k}"] = log_ret.shift(k)
feat["mom_5"] = np.log(px / px.shift(5))           # 1-week momentum
feat["mom_20"] = np.log(px / px.shift(20))         # 1-month momentum
feat["vol_20"] = log_ret.rolling(20).std()         # realized volatility over the last month
feat["range"] = (df["High"] - df["Low"]) / df["Close"]  # intraday range
feat["dist_ma50"] = px / px.rolling(50).mean() - 1      # distance from the 50-day moving average
feat["logvol_chg"] = np.log(df["Volume"]).diff()        # change in trading volume
feat["vix"] = df["VIX"]                                 # implied volatility level
feat["vix_chg"] = np.log(df["VIX"]).diff()              # change in implied volatility

feat.describe().T[["mean", "std", "min", "max"]]

## P6. Build the target: *tomorrow's* return

In [ ]:
next_ret = log_ret.shift(-1)   # return from close t to close t+1
data = feat.assign(next_ret=next_ret)
data[["ret_0", "next_ret"]].head(3)

> 🤔 **Think before moving on:** Check the table above: `next_ret` on row *t* should equal `ret_0` on row *t+1*. Why `shift(-1)` and not `shift(1)`? What would happen to your accuracy if you got the sign wrong?

In [ ]:
# Leakage smoke test: no feature should be strongly correlated with next-day return.
# A |corr| above ~0.2 on daily returns almost always means a lookahead bug.
corr = data.corr()["next_ret"].drop("next_ret").sort_values()
print(corr.round(3))
assert corr.abs().max() < 0.2, "Suspiciously strong correlation, so check for leakage!"

## P7. Drop warm-up rows and the last row

In [ ]:
print("Rows with NaN before dropna:", data.isna().any(axis=1).sum())
data = data.dropna()
print("Rows after dropna:", len(data), "|", data.index.min().date(), "->", data.index.max().date())

> 🤔 **Think before moving on:** Where do those NaN rows come from? Why is it fine to drop them here, but you must never `fillna(0)` on `next_ret`?

## P8. Chronological train/test split

In [ ]:
train = data.loc[:"2014"]
test = data.loc["2015":]

feature_cols = [c for c in data.columns if c != "next_ret"]
X_train, X_test = train[feature_cols], test[feature_cols]

y_train_up = (train["next_ret"] > 0).to_numpy()
y_test_up = (test["next_ret"] > 0).to_numpy()

print(f"train: {len(X_train)} days ({train.index.min().date()} -> {train.index.max().date()}), Up rate {y_train_up.mean():.3f}")
print(f"test:  {len(X_test)} days ({test.index.min().date()} -> {test.index.max().date()}), Up rate {y_test_up.mean():.3f}")

> 🤔 **Think before moving on:** Why not `train_test_split(X, y, test_size=0.2)` like in Chapter 2?

## P9. Cross-validation that respects time

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
for i, (tr_idx, va_idx) in enumerate(tscv.split(X_train)):
    print(f"fold {i}: train {X_train.index[tr_idx[0]].date()}..{X_train.index[tr_idx[-1]].date()}"
          f"  ->  validate {X_train.index[va_idx[0]].date()}..{X_train.index[va_idx[-1]].date()}")

> 🤔 **Think before moving on:** Use `tscv` wherever the book writes `cv=3`. Now try `cross_val_predict(model, X_train, y_train_up, cv=tscv)` and read the error it raises. Why does it fail? (Hint: which rows are never in a validation fold?) The helper below fixes this.

In [ ]:
from sklearn.base import clone

def oof_predict(model, X, y, cv, method="predict"):
    """Out-of-fold predictions for CV splitters that don't cover every row (e.g. TimeSeriesSplit).

    Returns (idx, preds): the row positions that were validated, and the predictions for them.
    Compare preds against y[idx].
    """
    y = np.asarray(y)
    all_idx, all_preds = [], []
    for tr_idx, va_idx in cv.split(X):
        fitted = clone(model).fit(X.iloc[tr_idx], y[tr_idx])
        all_preds.append(getattr(fitted, method)(X.iloc[va_idx]))
        all_idx.append(va_idx)
    return np.concatenate(all_idx), np.concatenate(all_preds)

## P10. Scaling: inside a `Pipeline`, never on the full dataset

SGD, SVC and KNN need scaled features. Fitting `StandardScaler` on all of `X_train` before CV would leak the validation folds' mean and std into training, so wrap the scaler with the model instead:

```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

sgd_clf = make_pipeline(StandardScaler(), SGDClassifier(random_state=42))
```

**✅ Data is ready.** You now have `X_train`, `X_test`, `y_train_up`, `y_test_up`, `tscv`, `oof_predict`, plus `train["next_ret"]` / `test["next_ret"]` for building other labels.

---
# Part 2: Your turn. Apply the Chapter 3 steps

Follow the order of `03_classification.ipynb`. Each heading names the book section to mirror. Wherever the book writes `cv=3`, use `tscv`, and use `oof_predict` instead of `cross_val_predict`.

## 1. Binary classifier + dummy baseline
*(Book: "Training a Binary Classifier", "Measuring Accuracy Using Cross-Validation")*

## 2. Confusion matrix, precision, recall

## 3. Precision/recall trade-off

## 4. ROC curve and AUC: SGD vs Random Forest

## 5. Multiclass: Down / Flat / Up
Hint: build the 3 classes from `train["next_ret"]`. Which data should you compute the cut points on?

## 6. Error analysis

## 7. Multilabel: [Up, BigMove]
Hint: define BigMove as |next-day return| > 1%. Compare the two labels with a metric that doesn't depend on the base rate.

## 8. Final evaluation on the test set (once!)

## 9. Your conclusion
Write 3–5 sentences: what can and can't be predicted here, and how you know your result isn't leakage or luck.